# Agentic Portfolio Construction — Full Pipeline Demo
**Fordham MSQF Capstone 2026**

5-agent pipeline: Research → Profile → Allocation ↔ Risk → Compliance

**Run Section 0 every time. Run Section 1 once to populate the data cache.**

---
## 0. Environment Setup
Run this cell first every session.

In [ ]:
import os, sys, importlib

# Notebook lives in notebooks/ — project root is one level up.
# We chdir to project root so agent output dirs (agents/research/, agents/profile/)
# resolve there instead of creating notebooks/agents/ which shadows our package.
_cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, "..")) if os.path.basename(_cwd) == "notebooks" else _cwd
os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

importlib.invalidate_caches()

print("Project root:", PROJECT_ROOT)
print("CWD:", os.getcwd())
print("FRED_API_KEY set:      ", bool(os.environ.get("FRED_API_KEY")))
print("ANTHROPIC_API_KEY set: ", bool(os.environ.get("ANTHROPIC_API_KEY")))

---
## 1. One-Time Data Cache Setup
**Skip this section if all files below show ✓.**

In [ ]:
from pathlib import Path

storage = Path("data/storage")
required = [
    "crsp_monthly.parquet",
    "crsp_daily.parquet",
    "ff_risk_factors.parquet",
    "fred_macro.parquet",
    "bls_oes.parquet",
    "ff12_monthly.parquet",
    "permno_map.json",
    "mkt_cap_weights.json",
]

all_present = True
for f in required:
    exists = (storage / f).exists()
    if not exists:
        all_present = False
    print(f"  {'OK' if exists else 'MISSING':<8} {f}")

if all_present:
    print("\nAll data cached. Jump to Section 2.")
else:
    print("\nMissing files — run the fetch cells below.")

In [ ]:
# WRDS — prompts for Fordham WRDS username + password on first run
from data.fetch.wrds import get_connection
conn = get_connection()
print("Connected to WRDS")

In [ ]:
from agents.allocation.adapters import DEFAULT_TICKERS
from data.fetch.wrds import fetch_crsp_monthly
fetch_crsp_monthly(DEFAULT_TICKERS, conn=conn)
print("crsp_monthly.parquet, permno_map.json, mkt_cap_weights.json  done")

In [ ]:
# Slow cell — 5-10 minutes
from data.fetch.wrds import fetch_crsp_daily
fetch_crsp_daily(DEFAULT_TICKERS, conn=conn)
print("crsp_daily.parquet  done")

In [ ]:
from data.fetch.wrds import fetch_ff_factors
fetch_ff_factors(conn=conn)
print("ff_risk_factors.parquet  done")

In [ ]:
from data.fetch.fred import fetch_fred_macro
fetch_fred_macro(fred_api_key=os.environ.get("FRED_API_KEY"))
print("fred_macro.parquet  done")

In [ ]:
from data.fetch.bls import fetch_bls_oes
fetch_bls_oes()
print("bls_oes.parquet  done")

In [ ]:
from data.fetch.factors import fetch_ff12
fetch_ff12()
print("ff12_monthly.parquet  done")

---
## 2. Research Agent — Macro Regime Detection

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

from agents.research.research_agent import run_research_agent

macro = run_research_agent(
    fred_api_key=os.environ.get("FRED_API_KEY"),
    run_comparison=False,
)

print(f"Current regime:  {macro.regime_label}")
print(f"Confidence:      {macro.regime_confidence:.0%}")
print(f"Prior regime:    {macro.prior_regime}")
print(f"Regime change:   {macro.regime_change_detected}")
print(f"Volatility:      {macro.regime_volatility:.4f}")

---
## 3. Profile Agent — Human Capital Valuation

In [ ]:
from agents.profile.profile_agent import run_profile_agent

# Builds all 9 BLS personas (p50 salary tier) in one pass
profiles = run_profile_agent(fred_api_key=os.environ.get("FRED_API_KEY"))

# SOC 15-1252 = Software Developers
profile = next(p for p in profiles if p.client_id == "bls_15-1252_p50")

print(f"Client:                {profile.client_id}")
print(f"Career type:           {profile.career_type}")
print(f"Financial capital:     ${profile.financial_capital:>12,.0f}")
print(f"Human capital (PV):    ${profile.human_capital_valuation:>12,.0f}")
print(f"Total wealth:          ${profile.total_wealth:>12,.0f}")
print(f"HC % of total:          {profile.human_capital_pct_of_total:.1f}%")
print(f"Income beta (β):        {profile.income_equity_beta:.2f}")
print(f"Income-equity corr (ρ): {profile.income_equity_correlation:.2f}")
print(f"Income volatility (σ):  {profile.income_volatility_sigma:.2f}")
print(f"Implicit equity exp:    {profile.implicit_equity_exposure:.1%}")
print(f"Effective risk budget:  {profile.effective_risk_budget:.1%}")
print(f"HC type:                {profile.human_capital_type.value}")
print(f"Risk tolerance:         {profile.risk_tolerance_level.value}")
print(f"Employer sector:        {profile.industry_exposure_sector}")
print(f"Investment horizon:     {profile.investment_horizon_years} years")

---
## 4. Full Pipeline — Allocation → Risk → Compliance

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S", force=True)

from agents.orchestrator.orchestrator import run_pipeline

pkg = run_pipeline(profile, macro, fred_api_key=os.environ.get("FRED_API_KEY"))
print("Pipeline complete.")

---
## 5. Results

In [ ]:
# Pipeline summary
m = pkg.metadata
print("=== PIPELINE SUMMARY ===")
print(f"Risk decision:      {m.final_risk_decision.value}")
print(f"Compliance status:  {m.final_compliance_status.value}")
print(f"Risk revisions:     {m.risk_revisions}")
print(f"Compliance revs:    {m.compliance_revisions}")
if m.pipeline_warnings:
    for w in m.pipeline_warnings:
        print(f"  Warning: {w}")

In [ ]:
# Portfolio weights
import pandas as pd

weights = pkg.allocation.proposed_portfolio
df_w = pd.DataFrame([
    {"Ticker": t, "Weight": w, "Weight %": f"{w:.1%}"}
    for t, w in sorted(weights.items(), key=lambda x: x[1], reverse=True)
    if w > 0.001
])
print("=== PORTFOLIO WEIGHTS ===")
print(df_w.to_string(index=False))

In [ ]:
# Risk metrics
r = pkg.risk
print("=== RISK METRICS ===")
print(f"Risk decision:   {r.risk_decision.value}")
if getattr(r, 'portfolio_volatility_annual', None):
    print(f"Volatility:      {r.portfolio_volatility_annual:.2%}  (annualized)")
print(f"Violations:      {r.violations or 'None'}")

print("\nRegime stress tests:")
for regime, ev in r.regime_evaluation.items():
    status = "PASS" if ev.passed else "FAIL"
    print(f"  [{status}] {regime:<35}  portfolio {ev.portfolio_drawdown:.1%}  bench {ev.benchmark_drawdown:.1%}")

In [ ]:
# Compliance results
c = pkg.compliance
print("=== COMPLIANCE ===")
print(f"Clearance:      {c.clearance}")
print(f"Status:         {c.compliance_status.value}")
print(f"Recommendation: {c.recommendation}")

print(f"\nPassed checks: {', '.join(c.passed_checks) if c.passed_checks else 'None'}")

if c.violations:
    print("\nViolations:")
    for v in c.violations:
        print(f"  [{v.severity.value}] {v.check_id}: {v.message}")

In [ ]:
# Allocation rationale (LLM-generated, one ticker as sample)
rationale = pkg.allocation.allocation_rationale
sample_ticker = next(iter(rationale))
print(f"=== ALLOCATION RATIONALE — {sample_ticker} ===")
print(rationale[sample_ticker])